In [108]:
import pandas as pd
import seaborn as sns
import matplotlib as plt
from pathlib import Path

# путь к файлу (в той же папке проекта)
DATA_PATH = Path.cwd() / "data3.csv"

# загрузка датасета
data3 = pd.read_csv(DATA_PATH)

In [117]:
#теперь нам необходимо немного подготовиться, перед тем как строить модель.
#идея: через one-hot кодирование аггрегированных групп признаков с расставлением флага продуктовый / персональный

# копия исходного датасета
# копия исходного датасета
data_enc = data3.copy()

# группы признаков
prod_features = ['company', 'product', 'review_source']
per_features = ['segment_name', 'age_segment', 'gender_cd', 'influencer_flg']

# переводим все признаки в строку,
# чтобы get_dummies закодировал даже ранговые/числовые
data_enc[prod_features + per_features] = (
    data_enc[prod_features + per_features]
    .astype(str)
)

# one-hot для product
prod_dummies = pd.get_dummies(
    data_enc[prod_features],
    prefix='prod',
    drop_first=True,
    dtype=int
)

# one-hot для personal
per_dummies = pd.get_dummies(
    data_enc[per_features],
    prefix='per',
    drop_first=True,
    dtype=int
)

# удаляем исходные колонки
data_enc.drop(
    columns=prod_features + per_features,
    inplace=True
)

# добавляем новые признаки
data_enc = pd.concat(
    [data_enc, prod_dummies, per_dummies],
    axis=1
)

# списки новых колонок
one_prod_cols = prod_dummies.columns.tolist()
one_per_cols = per_dummies.columns.tolist()

# проверка
print(data_enc.head())

   Unnamed: 0.1  Unnamed: 0          review_dttm          finish_dttm  \
0             0           0  2025-02-18 15:41:00  2025-02-18 16:56:49   
1             2           2  2025-07-08 07:40:43  2025-07-08 10:29:04   
2             3           3  2025-08-07 22:51:48  2025-08-08 09:35:34   
3             4           4  2025-02-13 21:38:40  2025-02-14 08:04:44   
4             5           5  2025-02-24 17:48:47  2025-02-24 18:34:06   

                          id_client  review_mark review_emotion  \
0  fb30834209a9c7f60612c64b82c75ffa            1     Негативный   
1  f1f8eff66eaf2289f61deec744871d6b            5     Позитивный   
2  5ca669878eaf593f68c10e163246357b            5     Позитивный   
3  7a436100b113ce78c8a7a02974521a16            5     Позитивный   
4  e60a8b542d1ffdfd980e0cd330f9d382            5     Позитивный   

         business_line         reason           review_theme  ... per_1 per_2  \
0      кредитные карты  Не определено       тарифы и условия  ...     0     1

In [71]:
#все сохраняем

data_enc = data_enc.copy()
data_enc.to_csv('data_enc.csv', index=False)

In [72]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

In [110]:
# путь к файлу (в той же папке проекта)
DATA_PATH = Path.cwd() / "data_enc.csv"

# загрузка датасета
data_enc = pd.read_csv(DATA_PATH)

In [129]:
import statsmodels.api as sm

print(data_enc.shape)

import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

y = data_enc['review_bin'].astype(int)

# сплитим выборку на тест / трэйн

train_idx, test_idx = train_test_split(
    data_enc.index,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# модель по продукт признакам

min_obs = 20

# удаляем редкие признаки
prod_cols_filtered = [
    col for col in one_prod_cols
    if data_enc[col].sum() >= min_obs
]

X_prod = data_enc[prod_cols_filtered].astype(int)

# удаляем константные колонки
X_prod = X_prod.loc[:, X_prod.nunique() > 1]

# train / test
X_prod_train = X_prod.loc[train_idx]
X_prod_test = X_prod.loc[test_idx]

y_train = y.loc[train_idx]
y_test = y.loc[test_idx]

# intercept
X_prod_train = sm.add_constant(X_prod_train)
X_prod_test = sm.add_constant(X_prod_test)

# модель
model_prod = sm.Logit(
    y_train,
    X_prod_train
).fit_regularized()

print(model_prod.summary())

# predictions
prod_pred = (
    model_prod.predict(X_prod_test) >= 0.5
).astype(int)

# accuracy
prod_acc = accuracy_score(y_test, prod_pred)

print(f'Product model accuracy: {prod_acc:.4f}')

#модель персональные признаки

# удаляем редкие признаки
per_cols_filtered = [
    col for col in one_per_cols
    if len(data_enc[col]) >= min_obs
]

X_per = data_enc[per_cols_filtered].astype(int)

# удаляем константные колонки
X_per = X_per.loc[:, X_per.nunique() > 1]

# удаляем дублирующиеся признаки (иначе singular matrix)
X_per = X_per.T.drop_duplicates().T

X_per_train = X_per.loc[train_idx]
X_per_test = X_per.loc[test_idx]

#включение констант
X_per_train = sm.add_constant(X_per_train)
X_per_test = sm.add_constant(X_per_test)

# модель
model_per = sm.Logit(
    y_train,
    X_per_train
).fit_regularized()

print(model_per.summary())

#предсказания
per_pred = (
    model_per.predict(X_per_test) >= 0.5
).astype(int)

#точность
per_acc = accuracy_score(y_test, per_pred)

print(f'Personal model accuracy: {per_acc:.4f}')

(27703, 74)
Optimization terminated successfully    (Exit mode 0)
            Current function value: 0.44431886868435877
            Iterations: 429
            Function evaluations: 429
            Gradient evaluations: 429
                           Logit Regression Results                           
Dep. Variable:             review_bin   No. Observations:                22162
Model:                          Logit   Df Residuals:                    22124
Method:                           MLE   Df Model:                           37
Date:                Tue, 26 May 2026   Pseudo R-squ.:                  0.2400
Time:                        23:11:19   Log-Likelihood:                -9847.0
converged:                       True   LL-Null:                       -12956.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                   coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------

In [100]:
#с учетом высокой вариативности человеческого поведения, мы получили неплохие оценки точности: 78,5% и 72,6%. Однако внутри модели очень раличаются

In [101]:
#теперь давайте оценим насколько наши модели устойчивы и какая из них лучше. в первой модели есть p-value = 0,99 у некоторых признаков, но это не означает, что модель непригодна сама по себе. проверим дополнительно с ROC-AUC

In [ ]:
#полученные тесты позволяют говорить о статистической значимости полученных результатов. гипотеза доказана.